# 1-Generate Predictions using LangChain

- **Goal:** Use LLMs to generate textual predictions for the following domains: financial, health, policy, weather, sports, and miscellaneous. 

- **Code Structure:** 

    1. Base template: This is included in every domain's input.
    2. Domain template: Vary or specific to a domain.

- **Run Notebook:**

    1. See README.md for installation and initial setup.
    2. Choose models (from `text_generation_models.py`) to generate data.
    3. Here in JupyerNotebook, click `Run All` button at top/in menu bar.
    4. Reach out if any problems.

In [2]:
import os
import sys

import pandas as pd

from langchain_core.prompts import PipelinePromptTemplate, PromptTemplate

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))

from log_files import LogData
from data_processing import DataProcessing
from text_generation_models import TextGenerationModelFactory

## Base Template for Domain Predictions

- `{prediction_properties}:` These are the variables for each prediction.
- `{prediction_requirements}:` These are to state how outcome predictions should be limited to or expressed as.
- `{prediction_templates}:` These are to give LLMs proper structure/syntax.
- `{prediction_examples}:` These are to provide LLMs with examples that match the templates.

In [3]:
full_prediction_template = """{prediction_properties}

{prediction_requirements}

{prediction_templates}

{prediction_examples}
"""
full_prediction_prompt = PromptTemplate.from_template(full_prediction_template)

In [4]:
prediction_properties_template = """A prediction <p> = (<p_s>, <p_t>, <p_d>, <p_o>), where it consists of the following four properties:

    1. <p_s>, any source entity in the {prediction_domain} domain.
        - Can be a person (with a name) or a {prediction_domain} person such as a {prediction_domain} reporter, {prediction_domain} analyst, {prediction_domain} expert, {prediction_domain} top executive, {prediction_domain} senior level person, etc), civilian.
        - Can only be an organization that is associated with the {prediction_domain} prediction.
    2. <p_t>, any target entity in the {prediction_domain} domain.
	    - Can be a person (with a name) or a {prediction_domain} person such as a {prediction_domain} reporter, {prediction_domain} analyst, {prediction_domain} expert, {prediction_domain} top executive, {prediction_domain} senior level person, etc).
        - Can only be an organization that is associated with the {prediction_domain} prediction.
    3. <p_d>, date or time range when <p> is expected to come to fruition or when one should observe the <p>.
        - Forecast can range from a second to anytime in the future.
        - Answers the questions: "How far to go out from today?" or "Where to stop?".
    4. <p_o>, {prediction_domain} prediction outcome.
        - Details relevant details such as outcome, a quantifiable metric, or slope.
        - Some examples are {prediction_domain_outcome}.
"""
prediction_properties_prompt = PromptTemplate.from_template(prediction_properties_template)

In [5]:
prediction_requirements_template = """{prediction_domain} requirements to use for each prediction:

    - Should be based on real-world {prediction_domain} data and not hallucinate.
    - Only a simple sentence (prediction) (and NOT compounding using "and" or "or").
    - Should diversify all four properties of the prediction (<p>) as in change and not use same for <p_s>, <p_t>, <p_d>, <p_o>.
    - Should use synonyms to predict such as forecasts, speculates, foresee, envision, etc., and not use any of them more than ten times.
    - The prediction should be unique and not repeated.
    - Do not number the predictions.
    - Do not say, "As the {prediction_domain}, I will generate company-based {prediction_domain} predictions using the provided templates." or anything similar.
    - Use the five different templates and examples provided.
    - Change how the current date (<p_d>) written in the prediction with examples of (1) Wednesday, August 21, 2024; (2) Wed, August 21, 2024; (3) 08/21/2024; (4) 08/21/2024; (5) 21/08/2024; (6) 21 August 2024; (7) 2024/08/21; (8) 2024-08-21; (9) August 21, 2024; (10) Aug 21, 2024; (11) 21 August 2024, (12) 21 Aug 2024, Q3 of 2027, 2029 of Q3, etc (with removing day of week).
    {domain_requirements}
    - Do not say, "Here are {predictions_N} unique {prediction_domain} predictions based on the provided templates and examples:" in the prompt.
    - Do not use any of the examples in the prompt.
    - In front of every prodiction, put the template number in the format of "T1:", "T2:", etc. and do not number them like "1.", "2.", etc. Should have template number and generated prediction matching.
    - Do not put template number on line by itself. Always pair with a prediction.
    - Disregard brackets: "<>"
    - Should never say "Here are {predictions_N} unique {prediction_domain} predictions based on the provided templates and examples:" or "Note: I've made sure to follow the guidelines and templates provided, and generated unique predictions that meet the requirements."
    - Do not use person name of entity name more than once as in don't use name Joe as both the <p_s> and <p_t>, unless like Mr. Sach and Goldman Sach or Mr. Sam Walton and Sam's Club, etc.
    - The source entity (<p_s>) is rarely the same as the target entity (<p_t>) and if same, the <p_s> is making a prediction on itself in the <p_t>.
    - Should variate the slope of rise/increase/as much as, fall/decrease/as little as, change, stay stable, high/low chance/probability/degree of, etc.
    - Should variate the prediction verbs such as will, would, be going to, should, etc.
"""
prediction_requirements_prompt = PromptTemplate.from_template(prediction_requirements_template)

In [6]:
prediction_templates_template = """Here are some {prediction_domain} templates:

    - {prediction_domain} template 1: <p_s> forecasts that the <p_o> at <p_t> potentially decrease in <p_d>.
    - {prediction_domain} template 2: On <p_d>, <p_s> speculates the <p_o> at <p_t> will likely increase.
    - {prediction_domain} template 3: <p_s> predicts on <p_d>, the <p_t> <p_o> may rise.
    - {prediction_domain} template 4: According to <p_s>, the <p_o> at <p_t> would fall in <p_d>.
    - {prediction_domain} template 5: In <p_d>, <p_s> envisions that <p_t> <p_o> has some probability to remain stable.
    - {prediction_domain} template 6: <p_t> <p_o> should stay same <p_d>, according to <p_s>. 
"""
prediction_templates_prompt = PromptTemplate.from_template(prediction_templates_template)

In [7]:
prediction_examples_template = """Here are some examples of {prediction_domain} predictions:
{domain_examples}

With the above (prediction with four properties, requirements, templates, and examples), generate a unique set of {predictions_N} predictions per template following the examples. Think from the perspective of an {prediction_domain} analyst, expert, top executive, or senior level person and even a college student, professional, research advisor, etc."""
prediction_examples_prompt = PromptTemplate.from_template(prediction_examples_template)

In [8]:
prediction_input_prompts = [
    ("prediction_properties", prediction_properties_prompt),
    ("prediction_requirements", prediction_requirements_prompt),
    ("prediction_templates", prediction_templates_prompt),
    ("prediction_examples", prediction_examples_prompt),
]
pipeline_prompt = PipelinePromptTemplate(
    final_prompt=full_prediction_prompt, pipeline_prompts=prediction_input_prompts
)

C:\Users\adria\AppData\Local\Temp\ipykernel_48288\2065471046.py:7: LangChainDeprecationWarning: This class is deprecated in favor of chaining individual prompts together.
  pipeline_prompt = PipelinePromptTemplate(


## Specific Templates for Domain Predictions

- For now, generating 1 prediction per template. From here, I'll try 3 and increase by increments/multiples of 3.

- With 1 prediction per template,
    - 1 prediction per template x 6 examples per domain so 6 predictions per domain
    - 6 predictions per domain x 4 domains = 24 predictions per model
    - 24 predictions per model x 2 models = 48 predictions across all models
    - 48 predictions across all models x 2 batches = 96 across all batches

In [9]:
examples_per_template = 1
generate_N_predictions_per_template = 1 * examples_per_template

### Template for Financial Predictions

In [10]:
financial_outcomes = """stock price, net profit, revenue, operating cash flow, research and development expenses, operating income, gross profit."""
financial_requirements = """- Should be based on real-world financial earnings reports.
   - Suppose the time when $p$ was made is during any earning season.
   - Include stocks from all sectors such as consumer staples, energy, finance, health care, industrials, materials, media, real estate, retail, technology, utilities, defense, etc.
   - Include the US Dollar sign ($) before or USD after the amount of the financial outcome."""

financial_examples = """
   - financial examples for template 1:
      1. Sebastian, an analyst forecasts that the stock price at Microsoft will decrease in 2026 Q2.
      2. Elena Rodriguez predicts that the net profit margin at Chevron should decrease in 01/10/2026 to 06/10/2026.
      3. Sebastian, an analyst predicts that the bonds he has will increase in 9/12/22.
   - financial examples for template 2:
      1. On July 20, 2026 to July 21, 2027, Barclays speculates that the inflation rates at the European Central Bank could rise.
      2. On May 14, 2010, Deutsche Bank forecasts that the asset value at Amazon should fall.
      3. On 8/15/2018, HSBC analysts foresee that their portfolio prices may increase.
   - financial examples for template 3:
      1. Citigroup anticipates that on 5/12/2026, the S&P 500 index could climb moderately.
      2. Vanguard foresees that on October 11, 2026, the price of Ethereum has a high probability of rising sharply.
      3. Citigroup predicts that on the 18 of November, 2026, there stock price might rise.
   - financial examples for template 4:
      1. According to UBS, the expected returns at real estate trusts could fall in August 2038.
      2. According to Sarah, the projected revenue at Alphabet Inc. should fall in Q4 2026.
      3. According to Microsoft, the trading volume it has will increase in 3/11/2021.
   - financial examples for template 5:
      1. In 9/25/2026, Santander expects that gold futures yields might stay stable.
      2. In 2/2/2026, David envisions that the dividend yield in Santander could stay stable.
      3. In 12/12/2027, John predicts that the crypto he has will stay stable.
   - financial examples for template 6:
      1. Google stock price will decrease in December 2026, according to Marcus.
      2. The FTSE 100 index is expected to rise in October 2026, according to Credit Suisse.
      3. Marcus foresees the asset price increasing in January 2027, according to his projections.
"""
financial_input_dict = {
    "prediction_domain": "financial",
    "prediction_domain_outcome": financial_outcomes,
    "domain_requirements": financial_requirements,
    "domain_examples": financial_examples,
    "predictions_N": generate_N_predictions_per_template
}
financial_prompt_output = pipeline_prompt.format(**financial_input_dict)
print(financial_prompt_output)


A prediction <p> = (<p_s>, <p_t>, <p_d>, <p_o>), where it consists of the following four properties:

    1. <p_s>, any source entity in the financial domain.
        - Can be a person (with a name) or a financial person such as a financial reporter, financial analyst, financial expert, financial top executive, financial senior level person, etc), civilian.
        - Can only be an organization that is associated with the financial prediction.
    2. <p_t>, any target entity in the financial domain.
	    - Can be a person (with a name) or a financial person such as a financial reporter, financial analyst, financial expert, financial top executive, financial senior level person, etc).
        - Can only be an organization that is associated with the financial prediction.
    3. <p_d>, date or time range when <p> is expected to come to fruition or when one should observe the <p>.
        - Forecast can range from a second to anytime in the future.
        - Answers the questions: "How fa

###  Template for Health Predictions

In [11]:
health_outcomes = """obesity rates, prevalence of chronic illnesses, average physical activity levels, nutritional intake, etc."""
health_requirements = """- Should be based on real-world health reports.
    - Suppose the time when $p$ was made is during any season such as flu season, allergy season, pandemic, epidemic, etc.
    - Include reports from all Health organization, researcher, doctor, physical therapist, physician assistant, nurse practictioners, fitness expert, etc."""
health_examples = """
    - health examples for template 1:
        1. She predicts that the smoking rates at the regional level should fall in late 2026.
        2. WHO forecasts that the incidence of respiratory infections at community clinics will decrease in Q1 2031.
        3. Julian thinks that the average daily step counts he does may increase in 2038.
    - health examples for template 2:
        1. On 5/12/2018, the Mayo Clinic speculates that the mental health wellness scores at European universities will change.
        2. On September 10, 2040, Leo speculates that the fiber intake at suburban households in Canada may increase.
        3. On the 15th of October, 2026, Leo predicts that the hypertension rates at the state level might decrease.
    - health examples for template 3:
        1. The NHS predicts that on 8/14/2019, participation in routine blood pressure checks will decline.
        2. Chloe suspects that on March 20, 2080, the diabetes prevalence at the global level could decrease.
        3. Silas envisions that on 12, November 2026, the average sleep duration for him potentially change.
    - health examples for template 4:
        1. According to the Research conducted at Oxford, the sodium intake at UK secondary schools might fall in Spring 2036.
        2. According to the Report, the average caffeine consumption at workplace environments will fall in 11/5/2026.
        3. According to Miller Study of heart health, the average cardiovascular endurance levels for him could fall in Q4, 2030.
    - health examples for template 5:
        1. In 2/15/2027, Dr. Hiroshi Tanaka envisions that local vaccination rates could increase.
        2. In July 2058, Professor Clara Smith envisions that average vegetable consumption among children will increase.
        3. In Q2 2027, Dr. Omar Kassab envisions that his hydration levels can potentially stay stable.
    - health examples for template 6:
        1. Vitamin D levels among office workers could fall in 3/22/2017, according to Dr. Sofia Rossi study.
        2. Mental health awareness in South America will become more apparent in the 12th of June 2016, according to the Report.
        3. Lucas's annual check-up frequency should rise in late 2025 Q4, according to Lucas.
"""
health_input_dict = {
    "prediction_domain": "health",
    "prediction_domain_outcome": health_outcomes,
    "domain_requirements": health_requirements,
    "domain_examples": health_examples,
    "predictions_N": generate_N_predictions_per_template
}
health_prompt_output = pipeline_prompt.format(**health_input_dict)

###  Template for Policy Predictions

In [12]:
policy_outcomes = """election outcomes, economic reforms, legislative impacts."""
policy_requirements = """- Should be based on real-world policy reports.
   - Suppose the time when $p$ was made is during an election cycle or non-election cycles.
   - Include policies & laws, from all sectors such as consumer staples, energy, finance, health care, industrials, materials, media, real estate, retail, technology, utilities, defense, etc."""

policy_examples = """
   - policy examples for template 1:
      1. Infrastructure spending in rural provinces should rise in national importance in the 10th of May 2014, according to analyst Simon Vance.
      2. Education reforms in the primary sector will increase in visibility in Q1 2026, according to Dr. Julian Thorne from the Heritage Foundation.
      3. She forecasts that cybersecurity protocols in the banking sector could stay stable in the public, in late 2027, according to expert Sofia Mendez.
   - policy examples for template 2:
      1. On 8/12/2030, the Heritage Foundation speculates that labor participation at industrial zones might stay stable.
      2. On July 22, 2022, the World Trade Organization speculates that trade tariffs at global markets will rise.
      3. On the 15th of March, 2026, policy analyst Victor Hugo speculates that antitrust enforcement in Victors media conglomerates may decrease.
   - policy examples for template 3:
      1. Representative Claire Walsh predicts that on December 12, 2028, the foreign aid allocation potentially can be stay stable.
      2. Economist Dr. Aris Thorne predicts that on 9/10/2026, the carbon tax rate in the industrial sector should rise.
      3. Senator Diana Prince predicts that on 7, August 2026, her constituent approval in urban districts will fall.
   - policy examples for template 4:
      1. According to Senator Robert Chen, the judicial transparency at state courts may increase in late 2018.
      2. According to Fatima Al-Sayed, the consumer confidence at the automotive sector will fall in Q2 2038.
      3. According to policy advisor Liam O'Connor, the retention rate at his government agencies shall stay stable in October 2026.
   - policy examples for template 5:
      1. In 11/15/2024, Senator Grace Hopper envisions that housing subsidies could stay stable.
      2. In April 2027, economist Dr. Kenji Sato envisions that interest rates in the commercial property sector will increase.
      3. In Q4 of 2023, policy strategist Elena Rossi envisions that her grant funding approvals may decline.
   - policy examples for template 6:
      1. Green hydrogen incentives are expected to inflate in Q2 2026, according to Dr. Isaac Newton.
      2. Agricultural grants will decrease in November 2060, according to Senator Sarah Jenkins.
      3. My candidate support in primary elections may stay the same in 05/04/2023, according to Carlos Santana.
"""
policy_input_dict = {
    "prediction_domain": "policy",
    "prediction_domain_outcome": policy_outcomes,
    "domain_requirements": policy_requirements,
    "domain_examples": policy_examples,
    "predictions_N": generate_N_predictions_per_template
}
policy_prompt_output = pipeline_prompt.format(**policy_input_dict)

###  Template for Weather Predictions

In [13]:
weather_outcomes = """temperature, precipitation, wind speed, humidity, etc."""
weather_requirements = """- Should be based on real-world weather reports.
    - Suppose the time when $p$ was made is during any season and any location (ie: Florida known for hurricanes, California known for wildfires, etc).
    - Include reports from all meteorologists, weather organizations, or any type of weather entity."""
weather_examples = """
    - weather examples for template 1:
        1. The World Meteorological Organization forecasts that the precipitation levels at London may increase in May 12, 2026.
        2. WeatherUnderground forecasts that the humidity at Austin will stay stable in early spring 2026.
        3. Erik a weather expert forecasts that the wind speed at his ranch could decrease in 8/14/2026.
    - weather examples for template 2:
        1. On 4/10/2026, Meteorologist Sarah Chen speculates that the temperature at Toronto will rise.
        2. On 2026 of December, Dr. Alan Grant speculates that the humidity at Dallas shall become lower.
        3. On August 15, 2026, Tokyo's meteorological team speculates that the wind speed in Tokyo potentially could stay stable.
    - weather examples for template 3:
        1. Elena Rossi, PH.D predicts that on November 2026, the temperature at Berlin can decrease.
        2. Meteorologist David Miller predicts that on September 1, 2026, the wind speed at Chicago will rise.
        3. The Sydney Weather Bureau predicts that on 12/2/2026, the humidity at Sydney might stay the same.
    - weather examples for template 4:
        1. According to Dr. Marcus Thorne, the temperature at Oslo shall increase in February 2027.
        2. According to Meteorologist Chloe Simmons, the precipitation levels at Vancouver will stay stable in March 1, 2026.
        3. According to Bruno, the wind speed at Brunos cabin may fall in 06/18/2026.
    - weather examples for template 5:
        1. In 10, January 2027, Meteorologist Julia Vance envisions that the temperature at Montreal could stay increase.
        2. In 9/22/2026, Dr. Steven Wu envisions that the humidity at Las Vegas will stay stable.
        3. In the month of October, the Madrid Weather Bureau envisions that the precipitation levels in Madrid should fall.
    - weather examples for template 6:
        1. Temperature in Cairo has a low chance to rise in August 2026, according to Meteorologist Amira Zahra.
        2. Humidity in Mumbai will decline in 2026, June 15, according to Dr. Thomas Fischer.
        3. Wind speed in Lisbon could stay stable in 11/10/2026, according to the Lisbon Weather Bureau.
"""
weather_input_dict = {
    "prediction_domain": "weather",
    "prediction_domain_outcome": weather_outcomes,
    "domain_requirements": weather_requirements,
    "domain_examples": weather_examples,
    "predictions_N": generate_N_predictions_per_template
}
weather_prompt_output = pipeline_prompt.format(**weather_input_dict)

### Template for Sports Predictions

In [14]:
sport_outcomes = """score, touchdown, goal, points, win, lose, etc."""
sport_requirements = """- Should be based on real-world sports.
    - Suppose the time when $p$ was made is during any season of sports.
    - Include reports from all sports professionals, coaches, or any type of sport entity."""
sport_examples = """
    - sport examples for template 1:
        1. Coach Andre Silva predicts that the touchdown rate at the San Francisco 49ers will fall in 2019 of October.
        2. Analyst Rebecca Hall forecasts that the goal average at FC Barcelona will stay the same in November 2027.
        3. Victor forecasts win percentage he has for tennis will go up in 05/20/2017.
    - sport examples for template 2:
        1. On Jan 15, 2105, Coach Isabella Conti suggests that the score average at the Los Angeles Lakers is climbing.
        2. On 11/04/2026, Analyst Simon Webb anticipates the touchdown rate at the Philadelphia Eagles will likely surge.
        3. On March 12, 2128, Thiago foresees that the win probability he has for cricket is expected to trend downward.
    - sport examples for template 3:
        1. Coach Sophia Loren predicts on 8/18/2026, the goal count at Manchester City will climb.
        2. Analyst Kenji Sato forecasts that on Sep 10, 2058, the point average at the Milwaukee Bucks will be higher.
        3. Leo Jr. estimates that on December 01, 2038, the win ratio for games he has will disimprove.
    - sport examples for template 4:
        1. According to Coach Paula Ferreira, the scoring average at the Boston Celtics is expected to dip in Sep 2023.
        2. According to Analyst Gary Vane, the touchdown rate at the Buffalo Bills will increase in 11/2026.
        3. According to Real Madrid, the win percentage at Real Madrid is projected to drop in October 2036.
    - sport examples for template 5:
        1. In 10/2026, Coach Hans Muller envisions that the goal average at Bayern Munich will hold steady.
        2. In October 2059, Analyst Maya Patel anticipates that the win rate at the Miami Heat will decrease slightly.
        3. In Sep 2088, Derek foresees that the points per game he has in basketball will gradually increase.
    - sport examples for template 6:
        1. The goal count at Arsenal FC will surge in Sep 2018, according to Coach Fabio Grosso.
        2. The win percentage at the New York Giants will taper off in October 2026, according to Analyst Chloe Zhang.
        3. The scoring average on Lucas's baseball team will remain steady in 09/2035, according to Lucas.
"""
sport_input_dict = {
    "prediction_domain": "sport",
    "prediction_domain_outcome": sport_outcomes,
    "domain_requirements": sport_requirements,
    "domain_examples": sport_examples,
    "predictions_N": generate_N_predictions_per_template
}
sport_prompt_output = pipeline_prompt.format(**sport_input_dict)

### Template for Miscellaneous Predictions

In [15]:
miscellaneous_outcomes = """These outcomes will take in any random outcome relating ot any real world situation."""
miscellaneous_requirements = """These outcomes will take in any random outcome relating ot any real world situation.
    - Suppose the time when $p$ was made is during any season or part of the year.
    - Include any type of entity."""
miscellaneous_examples = """
    - miscellaneous examples for template 1:
        1. Professor Elena Vance forecasts that the enrollment numbers at Oakwood University will drop in August 2024.
        2. Chef Marco Pierre predicts that the acidity level in the restaurant’s sauces will decline in fall 2028.
        3. Boardgame expert Silas Thorne foresees that the chances of drawing a red card at game night will decrease in November 2027.
    - miscellaneous examples for template 2:
        1. On 5/12/1998, Professor Julian Reed speculates that the graduation rate at Westview Institute will go up.
        2. On October 14, 2022, Chef Isabella Rossi suggests that the smokiness in the brisket at The Grill House will rise.
        3. On December 22, 2027, Chess master Magnus K. speculates that the chance of a draw in the tournament will increase.
    - miscellaneous examples for template 3:
        1. Dr. Aris Thorne predicts that on Nov 10, 2030, the average research output at St. Jude College will rise.
        2. Coach Bill Walsh predicts that on 04/15/2012, the completion rate at the San Francisco 49ers will improve.
        3. Clara predicts that on February 15, 2028, the odds of pulling an ace from the deck will rise.
    - miscellaneous examples for template 4:
        1. According to Coach Rick Pitino, the free-throw percentage at Kentucky Wildcats will fall in March 2005.
        2. According to Chef Gordon Ramsay, the creaminess of the risotto at Hell's Kitchen will likely decrease in Winter 2018.
        3. According to game analyst Leo Miller, the chance of hitting a jackpot in the slot machine will trend upward in June 2029.
    - miscellaneous examples for template 5:
        1. In fall 2032, Professor Sarah Jenkins envisions that the attendance records at Northway School will stay stable.
        2. In 08/2014, Chef David Chang envisions that the umami level at Momofuku will remain steady.
        3. In April of 2027, Gemini envisions that the probability of a coin flip landing on heads will stay consistent.
    - miscellaneous examples for template 6:
        1. Academic literacy will improve in Spring 2030, according to Professor H.M.
        2. The crunchiness of the fried chicken will increase in July 2019, according to Chef Anthony Bourdain.
        3. JR's chances of rolling a double six will stay the same on 02/05/2024, according to JR.
"""
miscellaneous_input_dict = {
    "prediction_domain": "miscellaneous",
    "prediction_domain_outcome": miscellaneous_outcomes,
    "domain_requirements": miscellaneous_requirements,
    "domain_examples": miscellaneous_examples,
    "predictions_N": generate_N_predictions_per_template
}
miscellaneous_prompt_output = pipeline_prompt.format(**miscellaneous_input_dict)

## Generate Predictions

### Text Generation Models

In [ ]:
tgmf = TextGenerationModelFactory(temperature=0.3, top_p=1)

# Groq Cloud (https://console.groq.com/docs/overview)
#gemma_29b_generation_model = tgmf.create_instance('gemma2-9b-it') 
llama_318b_instant_generation_model = tgmf.create_instance('llama-3.1-8b-instant') 
llama_3370b_versatile_generation_model = tgmf.create_instance('llama-3.3-70b-versatile')  
# llama_guard_4_12b_generation_model = tgmf.create_instance('meta-llama/llama-guard-4-12b')  

text_generation_models_groqcloud = [llama_318b_instant_generation_model, llama_3370b_versatile_generation_model]

In [20]:
N_batches = 3
prediction_domains = [
    "finance",
    "health",
    "policy",
    "weather",
    "sport",
    "miscellaneous"
]
prediction_prompt_outputs = {
    "finance": financial_prompt_output,
    "health": health_prompt_output,
    "policy": policy_prompt_output,
    "weather": weather_prompt_output,
    "sport": sport_prompt_output,
    "miscellaneous": miscellaneous_prompt_output,
}

prediction_label = 1
save_logs = "data"
batched_predictions_df = tgmf.batch_generate_data(N_batches=N_batches, 
                                text_generation_models=text_generation_models_groqcloud, 
                                domains=prediction_domains,
                                prompt_outputs=prediction_prompt_outputs,
                                sentence_label=prediction_label,
                                save_path=save_logs)

  0%|          | 0/3 [00:00<?, ?it/s]

===================================== Batch 0 ===============================================
finance --- llama-3.1-8b-instant --- GROQ_CLOUD
generates:
T1: Sebastian, a financial analyst forecasts that the stock price at Apple will decrease in 2027 Q2.

T2: On August 21, 2024, Goldman Sachs speculates that the operating cash flow at Amazon will likely increase.

T3: Citigroup predicts on Wednesday, August 21, 2024, the S&P 500 index could climb moderately.

T4: According to Morgan Stanley, the expected returns at real estate trusts could fall in Q3 of 2027.

T5: In 2024-08-21, Santander expects that gold futures yields might stay stable.

T6: Google stock price will decrease in August 2024, according to Marcus.
finance --- llama-3.3-70b-versatile --- GROQ_CLOUD
generates:
T1: JPMorgan forecasts that the revenue at Boeing will potentially decrease in Q3 of 2027.
T2: On 2024-08-21, Goldman Sachs speculates that the stock price at Apple will likely increase.
T3: Morgan Stanley predicts o

 33%|███▎      | 1/3 [01:41<03:23, 101.58s/it]

===================================== Batch 1 ===============================================
finance --- llama-3.1-8b-instant --- GROQ_CLOUD
generates:
T1: Sebastian, a financial analyst forecasts that the stock price at Apple will decrease in 2027 Q4.
T2: On August 22, 2024, Goldman Sachs speculates that the operating cash flow at Amazon will likely increase.
T3: Vanguard predicts on September 15, 2026, the price of Tesla has a high probability of rising sharply.
T4: According to Morgan Stanley, the expected returns at the S&P 500 index could fall in 2028 Q2.
T5: In 2029 Q3, Citigroup envisions that the dividend yield in Johnson & Johnson could stay stable.
T6: Google stock price will remain the same in December 2026, according to Credit Suisse.
finance --- llama-3.3-70b-versatile --- GROQ_CLOUD
generates:
T1: JPMorgan forecasts that the revenue at Boeing will potentially decrease in Q3 of 2027.
T2: On August 15, 2024, to August 16, 2025, Goldman Sachs speculates that the stock price

 67%|██████▋   | 2/3 [03:25<01:43, 103.15s/it]

===================================== Batch 2 ===============================================
finance --- llama-3.1-8b-instant --- GROQ_CLOUD
generates:
T1: Citigroup forecasts that the operating cash flow at Apple will decrease in 2027 Q4.

T2: On August 22, 2024, Goldman Sachs speculates that the revenue at Amazon will likely increase.

T3: Vanguard predicts on September 15, 2026, the stock price at Tesla may rise.

T4: According to Morgan Stanley, the net profit at ExxonMobil would fall in Q2 2028.

T5: In Q3 2027, JPMorgan Chase envisions that the research and development expenses at Alphabet Inc. have some probability to remain stable.

T6: Google stock price will stay same in November 2026, according to Credit Suisse.
finance --- llama-3.3-70b-versatile --- GROQ_CLOUD
generates:
T1: JPMorgan forecasts that the revenue at Boeing will potentially decrease in Q3 of 2027.
T2: On 2024-08-21, Goldman Sachs speculates that the stock price at Apple will likely increase.
T3: Morgan Stanle

100%|██████████| 3/3 [05:24<00:00, 108.02s/it]


CSV to Log


In [21]:
pd.set_option('max_colwidth', 200)
#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)
batched_predictions_df.head()

,Base Sentence,Sentence Label,Domain,Model Name,API Name,Batch ID,Temperature,Top P,Generated At,Prompt Used,Template Number
0,"Sebastian, a financial analyst forecasts that the stock price at Apple will decrease in 2027 Q2.",1,finance,llama-3.1-8b-instant,GROQ_CLOUD,0,0.2,1,2026-02-28,"A prediction <p> = (<p_s>, <p_t>, <p_d>, <p_o>), where it consists of the following four properties:\n\n 1. <p_s>, any source entity in the financial domain.\n - Can be a person (with a ...",1
1,"On August 21, 2024, Goldman Sachs speculates that the operating cash flow at Amazon will likely increase.",1,finance,llama-3.1-8b-instant,GROQ_CLOUD,0,0.2,1,2026-02-28,"A prediction <p> = (<p_s>, <p_t>, <p_d>, <p_o>), where it consists of the following four properties:\n\n 1. <p_s>, any source entity in the financial domain.\n - Can be a person (with a ...",2
2,"Citigroup predicts on Wednesday, August 21, 2024, the S&P 500 index could climb moderately.",1,finance,llama-3.1-8b-instant,GROQ_CLOUD,0,0.2,1,2026-02-28,"A prediction <p> = (<p_s>, <p_t>, <p_d>, <p_o>), where it consists of the following four properties:\n\n 1. <p_s>, any source entity in the financial domain.\n - Can be a person (with a ...",3
3,"According to Morgan Stanley, the expected returns at real estate trusts could fall in Q3 of 2027.",1,finance,llama-3.1-8b-instant,GROQ_CLOUD,0,0.2,1,2026-02-28,"A prediction <p> = (<p_s>, <p_t>, <p_d>, <p_o>), where it consists of the following four properties:\n\n 1. <p_s>, any source entity in the financial domain.\n - Can be a person (with a ...",4
4,"In 2024-08-21, Santander expects that gold futures yields might stay stable.",1,finance,llama-3.1-8b-instant,GROQ_CLOUD,0,0.2,1,2026-02-28,"A prediction <p> = (<p_s>, <p_t>, <p_d>, <p_o>), where it consists of the following four properties:\n\n 1. <p_s>, any source entity in the financial domain.\n - Can be a person (with a ...",5
